In [4]:
%store -r train_df df validation_df test_df

In [5]:
#Z-score and IQR implementations

import numpy as np
from sklearn.metrics import precision_recall_curve

SEED = 42

feature_cols = [col for col in df.columns if col != 'Class']

X_train = train_df[feature_cols]
X_val = validation_df[feature_cols]
Y_val = validation_df["Class"]

In [ ]:
# reference stats on training data

train_mean = X_train.mean()
train_std = X_train.std(ddof=0)

#Handles features that are constant in the training data separately.
z_score = (X_val - train_mean).abs().div(train_std.replace(0, np.nan))
z_flags = z_score.gt(3) | (
    train_std.eq(0) & X_val.ne(train_mean)
)
z_score = z_flags.fillna(False).mean(axis=1)

#IQR fence implementation on training features only
q1 = X_train.quantile(0.25)
q3 = X_train.quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

iqr_flags = X_val.lt(lower_fence) | X_val.gt(upper_fence)
iqr_score = iqr_flags.mean(axis=1)


In [7]:
#Threshold selection using validation data

# This function calculates the best F1 score and its corresponding threshold based on the precision-recall curve.
def best_f1_threshold(y_true, scores): 
    precision, recall, thresholds = precision_recall_curve(y_true, scores)

    # Check if thresholds is empty and raise an error if it is
    if len(thresholds) == 0:
        raise ValueError("No thresholds found. Cannot select a threshold: validation scores have no threshold values.")

    #The final precision/recall values do not correspond to a threshold.
    f1 = (
        2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12) 
    )
    best_idx = np.argmax(f1)
    return f1[best_idx], thresholds[best_idx]

""" Selecting the best threshold for both Z-score and IQR methods based on the validation set. """
z_val_f1, z_threshold = best_f1_threshold(Y_val, z_score)
iqr_val_f1, iqr_threshold = best_f1_threshold(Y_val, iqr_score)

print(f"z-score threshold: {z_threshold:.4f}; validation F1: {z_val_f1:.4f}")
print(f"IQR threshold: {iqr_threshold:.4f}; validation F1: {iqr_val_f1:.4f}")


z-score threshold: 0.5333; validation F1: 0.2804
IQR threshold: 0.5667; validation F1: 0.3357


### *Notes about implementation* 
- _The notebook uses the fraud labels in the validation set to asses two simple anomaly-detection baselines: Z-score and IQR_

- _The `best_f1_threshold` function is used to find the cutoff score with the highest validation F1 for each method_

- _Issues faced was: trying to access the data frame with loading a new one from the start, how it was resolved was by using the `%store` to save the data frame from the first notebook._

In [10]:
import json
from pathlib import Path

import numpy as np
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

def evaluate_scores(y_true, scores, threshold):
    predictions = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true, predictions, labels=[0, 1]
    ).ravel()

    return {
        "threshold": float(threshold),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "f1": float(f1_score(y_true, predictions, zero_division=0)),
        "average_precision": float(average_precision_score(y_true, scores)),
        "false_positive_rate": float(fp / (fp + tn)) if (fp + tn) else 0.0,
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
    }

validation_results = {
    "z_score": evaluate_scores(Y_val, z_score, z_threshold),
    "iqr": evaluate_scores(Y_val, iqr_score, iqr_threshold),
}

# Save the training-fitted statistics too, so test scores use the same reference.
record = {
    "random_seed": 42,
    "feature_cols": feature_cols,
    "reference_statistics": {
        "mean": {key: float(value) for key, value in train_mean.items()},
        "std": {key: float(value) for key, value in train_std.items()},
        "q1": {key: float(value) for key, value in q1.items()},
        "q3": {key: float(value) for key, value in q3.items()},
    },
    "validation_results": validation_results,
}

current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name.casefold() == "notebooks" else current_dir
results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

record_path = results_dir / "statistical_baselines.json"
with record_path.open("w", encoding="utf-8") as file:
    json.dump(record, file, indent=2)

print(f"Saved thresholds and training statistics to {record_path.resolve()}")
print(json.dumps(validation_results, indent=2))

Saved thresholds and training statistics to C:\Users\user\Documents\AlgomaU courses\Computer Science\Third Year\Data Science with Python\Data Science Final Project\results\statistical_baselines.json
{
  "z_score": {
    "threshold": 0.5333333333333333,
    "precision": 0.45454545454545453,
    "recall": 0.20270270270270271,
    "f1": 0.2803738317757009,
    "average_precision": 0.17856191790609344,
    "false_positive_rate": 0.00042206954768213474,
    "true_negatives": 42629,
    "false_positives": 18,
    "false_negatives": 59,
    "true_positives": 15
  },
  "iqr": {
    "threshold": 0.5666666666666667,
    "precision": 0.34782608695652173,
    "recall": 0.32432432432432434,
    "f1": 0.3356643356643357,
    "average_precision": 0.20571779680586155,
    "false_positive_rate": 0.0010551738692053368,
    "true_negatives": 42602,
    "false_positives": 45,
    "false_negatives": 50,
    "true_positives": 24
  }
}
